# Gisborne screening: deriving the committed findings

This notebook recomputes each headline number in the project README directly
from the committed real-data outputs, so the claims can be checked without
rerunning the whole pipeline. It covers the three findings in README order:
overlay materiality, width-method disagreement, and the outstanding imagery
review. Neither width proxy is MPI's formal centre-line measurement.

In [1]:
from pathlib import Path
import json
import warnings

import geopandas as gpd
import pandas as pd

ROOT = Path('..')
OUT = ROOT / 'outputs' / 'gisborne'

candidates = gpd.read_file(ROOT / 'data/processed/gisborne_candidates.gpkg')
screened = pd.concat(
    [gpd.read_file(OUT / 'candidates.gpkg'), gpd.read_file(OUT / 'quarantine.gpkg')],
    ignore_index=True,
)
screened = gpd.GeoDataFrame(screened, geometry='geometry', crs=2193)
comparison = pd.read_csv(OUT / 'width_method_comparison.csv')
findings = json.loads((OUT / 'findings.json').read_text())

print(f'Input LCDB units:      {len(candidates):,}')
print(f'Screened dispositions: {len(screened):,}')
print(screened.status.value_counts().to_string())

Input LCDB units:      5,712
Screened dispositions: 5,712
status
candidate_review    3069
quarantine          2443
excluded             200


## Finding 1 — one relative threshold cannot judge overlap on units from 0.0001 to 36,801 ha

The project has used three materiality rules. The first flagged any
intersection over 1 m² and excluded whole units for generalisation slivers. The
second required 1% of the unit, which waved through hundreds of hectares of
conflict inside very large units as a “minor” advisory. The current rule
removes overlap narrower than 15 m before judging it, treats the remainder as
material at 1% of the unit, and marks any remaining overlap of 1 ha or more as
**clip-required** rather than as noise. The cell compares the three rules using
the pipeline's own published decision columns.

In [2]:
def overlay_sensitivity(prefix, label):
    # Reads the published decision columns instead of re-deriving materiality
    # from rounded values, so the notebook cannot disagree with the pipeline.
    raw = screened[f'{prefix}_overlap_m2']
    flagged = raw > 1.0
    pct_only = flagged & (screened[f'{prefix}_overlap_pct'] >= 1.0)
    material = screened[f'{prefix}_material']
    clip = screened[f'{prefix}_clip_required']
    advisory = flagged & ~material & ~clip
    print(f'{label}:')
    print(f'  area-only flags (>1 m2):              {int(flagged.sum()):,}')
    print(f'  material under the 1%-only rule:      {int(pct_only.sum()):,}')
    print(f'  material now (sliver-filtered, >=1%): {int(material.sum()):,}')
    print(f'  clip-required (>=1 ha, <1%):          {int(clip.sum()):,}')
    print(f'  low-overlap advisory:                 {int(advisory.sum()):,}')
    print(f'  overlap inside clip-required units:   {raw[clip].sum()/1e4:,.1f} ha')


overlay_sensitivity('conservation', 'R-04 conservation')
print()
overlay_sensitivity('pre1990', 'R-03 mapped pre-1990')

R-04 conservation:
  area-only flags (>1 m2):              302
  material under the 1%-only rule:      224
  material now (sliver-filtered, >=1%): 200
  clip-required (>=1 ha, <1%):          27
  low-overlap advisory:                 75
  overlap inside clip-required units:   673.2 ha

R-03 mapped pre-1990:
  area-only flags (>1 m2):              900
  material under the 1%-only rule:      691
  material now (sliver-filtered, >=1%): 603
  clip-required (>=1 ha, <1%):          44
  low-overlap advisory:                 253
  overlap inside clip-required units:   1,044.5 ha


Under the 1%-only rule, 48 units carried at least 1 ha of mapped pre-1990
planted forest (1,012.4 ha in total, up to 146.9 ha in one unit) and 25 units
carried at least 1 ha of DOC land (668.6 ha, up to 297.2 ha) while being reported
as minor overlaps. They were not slivers. Excluding those units whole would be
the opposite error — 25 DOC cases alone cover about 241,500 ha — so they stay in
review with a clip-required flag and their conflict-free area is published.

In [3]:
advisory = pd.read_csv(OUT / 'advisory_candidates.csv')
print(f'candidates carrying an advisory flag: {len(advisory)}')
print(advisory.advisory_rule_ids.value_counts().to_string())
print()
print(advisory.head(5).to_string(index=False))

candidates carrying an advisory flag: 560
advisory_rule_ids
R-01-contiguous                                     156
R-03-low-overlap                                    146
R-01-contiguous|R-02-contiguous                     125
R-01-near-threshold                                  39
R-04-low-overlap                                     24
R-03-clip-required                                   20
R-03-clip-required|R-04-clip-required                16
R-02-contiguous                                      11
R-03-clip-required|R-04-low-overlap                   5
R-03-low-overlap|R-04-clip-required                   4
R-01-contiguous|R-03-low-overlap                      3
R-01-contiguous|R-02-contiguous|R-03-low-overlap      3
R-03-low-overlap|R-04-low-overlap                     3
R-04-clip-required                                    2
R-01-near-threshold|R-03-low-overlap                  1
R-02-contiguous|R-03-low-overlap                      1
R-01-contiguous|R-02-contiguous|R-04-low-ove

## Finding 2 — the width result depends on the proxy

`2A/P` reads every shape narrower than it is: a W × L rectangle gives
WL/(W+L), so a 33 m wide, 1 ha strip reads 29.8 m. R-02 now uses the
equivalent-rectangle width (the rectangle with the same area and perimeter,
exact for any rectangle) together with the `-15 m` erosion core test.

In [4]:
ap_disagree = comparison.area_perimeter_pass != comparison.erosion_core_pass
core_only = comparison.erosion_core_pass & ~comparison.equivalent_rectangle_pass

print(f'2A/P vs erosion disagreements:        {int(ap_disagree.sum()):,} of {len(comparison):,} '
      f'({ap_disagree.mean():.2%})')
print(f'Rectangle width vs erosion (R-02):    {int(comparison.methods_disagree.sum()):,}')
print(f'  erosion core, rectangle below 30 m: {int(core_only.sum()):,}')
print()
print('Equivalent-rectangle width (m) where the R-02 proxies disagree:')
print(comparison.loc[core_only, 'width_equivalent_rectangle_m'].describe()[['min', '50%', 'max']].to_string())
print()
print({k: findings[k] for k in ('core_only_units_with_split_core',
                                 'core_only_units_with_two_parts_over_500_m2',
                                 'core_only_units_with_two_parts_over_1000_m2')})

2A/P vs erosion disagreements:        694 of 5,712 (12.15%)
Rectangle width vs erosion (R-02):    381
  erosion core, rectangle below 30 m: 381

Equivalent-rectangle width (m) where the R-02 proxies disagree:
min    13.781
50%    25.644
max    29.978

{'core_only_units_with_split_core': 126, 'core_only_units_with_two_parts_over_500_m2': 33, 'core_only_units_with_two_parts_over_1000_m2': 14}


![Three real Gisborne disagreement cases](../outputs/gisborne/figures/width_disagreement_cases.png)

The 694 `2A/P`-versus-erosion disagreements all ran one way because
2A/P ≤ inscribed diameter for every shape; 293 of them were convex, so most
came from the proxy's bias on compact shapes, not from branching outlines.
With the equivalent rectangle, 381 units keep a 30 m core but average below
30 m; 126 of those cores are split into separate fragments. All are
quarantined under R-02 unless contiguous with a qualifying unit.

## The CRS failure mode is false rejection

The original project brief predicted that a degrees-versus-metres mistake would
let small polygons through. The real direction is the opposite.

In [5]:
nztm_pass = candidates.area >= 10_000
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    naive_wgs84_pass = candidates.to_crs(4326).area >= 10_000

print(f'NZTM2000 >=1 ha:       {int(nztm_pass.sum()):,}')
print(f'Naive WGS84 >=1 ha:    {int(naive_wgs84_pass.sum()):,}')
print(f'False rejections:      {int((nztm_pass & ~naive_wgs84_pass).sum()):,}')
print(f'False qualifications:  {int((~nztm_pass & naive_wgs84_pass).sum()):,}')

NZTM2000 >=1 ha:       4,223
Naive WGS84 >=1 ha:    0
False rejections:      4,223
False qualifications:  0


All 4,223 true area passes become false rejections, because a square degree
compared against 10,000 is a vanishingly small number. Nothing is falsely
qualified. The finding corrects the initial hypothesis instead of being fitted
to it, and every pipeline entry point now refuses input that is not EPSG:2193.

## Finding 3 — the imagery review is still outstanding

R-05 tests the mapped land-cover class, not what is on the ground. The fixed
sample of 30 candidates is pinned and rendered as imagery cards, but no label
file exists yet, so the findings carry a pending status and no rate.

In [6]:
pinned = pd.read_csv(OUT / 'review/review_sample_ids.csv')
summary = pd.read_csv(OUT / 'review/review_summary.csv')

print(f'pinned review sample: {len(pinned)} units')
print(f'first three:          {", ".join(pinned.unit_id.head(3))}')
print()
print(summary.to_string(index=False))
print()
print({k: v for k, v in findings.items() if k.startswith('visual_')})

pinned review sample: 30 units
first three:          lcdb1000012117, lcdb1000034948, lcdb1000035223

         metric                                                                                                                                        value
    sample_size                                                                                                                                           30
  review_status                                                                                                             pending_independent_human_review
labels_recorded                                                                                                                                            0
        imagery                                                                 Gisborne District Council Imagery Satellite Gisborne 2024 (credited to LINZ)
how_to_complete Copy review_labels_template.csv to review_labels.csv, label all 30 cards from review/cards/, then 

Any agreement or false-positive rate published from this sample must come from
a named person working through `outputs/gisborne/review/cards/`;
`scripts/ingest_review_labels.py` rejects a label file whose reviewer looks
automated.

## Reading the unit counts

Multipart LCDB units are exploded before screening, and each fragment is
screened independently under its own `-part-N` identifier. A count in this
notebook is therefore a count of screened fragments, not of source LCDB
polygons, and fragments of one polygon are not re-aggregated before R-01 or
R-02 are applied.